# Workshop 3 – Science Theme: Generation with Retrieved Context

In this notebook, we move from **retrieval** to **generation**.

In Workshop 2, we built a retrieval system using scientific text data and stored it in a FAISS vector database.  
In this workshop, we will **load that saved index** and use it to retrieve relevant scientific content for a query.

We will then pass the retrieved context to the **Qwen model** and generate answers grounded in that context.

## Learning goals

By the end of this notebook, you should be able to:

- load a saved FAISS index from Workshop 2
- retrieve relevant scientific content for a query
- use retrieved context as input to an LLM
- generate grounded answers with Qwen
- reflect on the quality of generated responses

## Step 1: Install Required Libraries

We will use:

- **LangChain** to load the FAISS vector database
- **Transformers** to run the Qwen language model
- **Sentence Transformers** for embeddings (used when loading FAISS)
- **FAISS** for loading and retrieving embeddings from the vector database

These libraries were also used in previous workshops.

In [ ]:
#Installing required libraries
!pip install -q langchain langchain-community transformers sentence-transformers faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [ ]:
#Import libraries
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline

## Step 2: Load the Saved FAISS Index

In Workshop 2, we created a FAISS vector database using chunks from scientific articles.

Here, we load that saved index instead of rebuilding it from scratch.

This is a more practical workflow and saves time, especially when working with larger datasets.

In [ ]:
# Load the same embedding model used when creating the FAISS index
embedding_model = HuggingFaceEmbeddings()

# Load the saved FAISS index
db = FAISS.load_local(
    "faiss_Sci",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS index loaded successfully.")

#Note: If you see warnings during loading, they are usually safe to ignore as long as the index loads successfully.

/tmp/ipykernel_953/2288510738.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings()
/tmp/ipykernel_953/2288510738.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding_model = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/set

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index loaded successfully.


## Step 3: Test Retrieval

Before moving to generation, we first test whether retrieval is working correctly.

We will submit a science-related query and inspect the returned chunks.

In [ ]:
query = "neural networks for image classification"

results = db.similarity_search(query, k=5)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n")
    print(r.page_content[:700])


Result 1:

Support Vector Machines (SVMs). In this study it was observed that the ANNs
perform better than the SVMs. This performance is measured against the
generalisation ability of the two.

Result 2:

Comparing Robustness of Pairwise and Multiclass Neural-Network Systems
  for Face Recognition   Noise, corruptions and variations in face images can seriously hurt the
performance of face recognition systems. To make such systems robust,
multiclass neuralnetwork classifiers capable of learning from noisy data have
been suggested. However on large face data sets such systems cannot provide the
robustness at a high level. In this paper we explore a pairwise neural-network
system as an alternative approach to improving the robustness of face
recognition. In our experiments this approach is shown to outperform the
multiclass neural-network system in terms of the predictive accuracy on the
face i

Result 3:

The Parameter-Less Self-Organizing Map algorithm   The Parameter-Less Self-Organi

## Step 4: Prepare Context for Generation

The retrieved chunks will now be combined into one context block.

This context will be passed to the language model, so that the answer is based on relevant scientific content rather than relying only on the model’s general knowledge.

In [ ]:
# Combine the retrieved chunks into one context
context = "\n\n".join([r.page_content for r in results])

print(context[:1000])

Support Vector Machines (SVMs). In this study it was observed that the ANNs
perform better than the SVMs. This performance is measured against the
generalisation ability of the two.

Comparing Robustness of Pairwise and Multiclass Neural-Network Systems
  for Face Recognition   Noise, corruptions and variations in face images can seriously hurt the
performance of face recognition systems. To make such systems robust,
multiclass neuralnetwork classifiers capable of learning from noisy data have
been suggested. However on large face data sets such systems cannot provide the
robustness at a high level. In this paper we explore a pairwise neural-network
system as an alternative approach to improving the robustness of face
recognition. In our experiments this approach is shown to outperform the
multiclass neural-network system in terms of the predictive accuracy on the
face images corrupted by noise.

The Parameter-Less Self-Organizing Map algorithm   The Parameter-Less Self-Organizing Map 

## Step 5: Load the Qwen Model

We now load the same Qwen model used in Workshop 1.

Using the same model helps us focus on the new idea in this workshop:
using retrieved context to support answer generation.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

generator = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name
)

print("Model loaded successfully.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded successfully.


## Step 6: Generate an Answer from Retrieved Context

We now give the model:

- the retrieved context
- the question
- a short instruction

This is a simple example of grounded generation, where the model is encouraged to answer using the retrieved scientific text.

In [ ]:
prompt = f"""
You are a scientific assistant.

First, check if the context contains the answer.
If yes, answer using the context.
If not, return ONLY: I don't know.
Do not attempt to answer before checking.

Context:
{context}

Question:
What approaches are used for image classification in this context?

Answer:
"""

response = generator(prompt, max_new_tokens=200)

print(response[0]["generated_text"])

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a scientific assistant.

First, check if the context contains the answer.
If yes, answer using the context.

Context:
Support Vector Machines (SVMs). In this study it was observed that the ANNs
perform better than the SVMs. This performance is measured against the
generalisation ability of the two.

Comparing Robustness of Pairwise and Multiclass Neural-Network Systems
  for Face Recognition   Noise, corruptions and variations in face images can seriously hurt the
performance of face recognition systems. To make such systems robust,
multiclass neuralnetwork classifiers capable of learning from noisy data have
been suggested. However on large face data sets such systems cannot provide the
robustness at a high level. In this paper we explore a pairwise neural-network
system as an alternative approach to improving the robustness of face
recognition. In our experiments this approach is shown to outperform the
multiclass neural-network system in terms of the predictive accuracy on 

## Step 7: Augmenting the Model with Retrieved Knowledge

So far, we forced the model to use only the retrieved context.

Now, we relax this constraint.

The model can:
- use the retrieved context as the main source
- add general knowledge to improve clarity

This often produces more complete and natural answers, but may introduce some risk of hallucination.

In [ ]:
augmented_prompt = f"""
You are a scientific assistant.

Use the context below as your main source of information.
You may use your own knowledge to explain and clarify the answer.

Do not invent references.

Context:
{context}

Question:
What approaches are used for image classification in this context?

Answer:
"""

response_aug = generator(augmented_prompt, max_new_tokens=200)

print(response_aug[0]["generated_text"])

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a scientific assistant.

Use the context below as your main source of information.
You may use your own knowledge to explain and clarify the answer.

Do not invent references.

Context:
Support Vector Machines (SVMs). In this study it was observed that the ANNs
perform better than the SVMs. This performance is measured against the
generalisation ability of the two.

Comparing Robustness of Pairwise and Multiclass Neural-Network Systems
  for Face Recognition   Noise, corruptions and variations in face images can seriously hurt the
performance of face recognition systems. To make such systems robust,
multiclass neuralnetwork classifiers capable of learning from noisy data have
been suggested. However on large face data sets such systems cannot provide the
robustness at a high level. In this paper we explore a pairwise neural-network
system as an alternative approach to improving the robustness of face
recognition. In our experiments this approach is shown to outperform the
mult

## Reflection

Compare this answer with the previous strict version.

- Is the answer more detailed?
- Is it easier to understand?
- Does it include information not present in the context?
- Which version do you trust more?

This shows the trade-off between:
- strict grounding (more reliable)
- augmented generation (more expressive)